In [24]:
import os
import warnings
import logging

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
logging.getLogger("tensorflow").setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

import cv2
import shutil
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.applications import ResNet50, EfficientNetB0, DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

IMG_SIZE = 224
DATASET_PATH = "dataset"
MODEL_DIR = "models"
EPOCHS = 10
BATCH_SIZE = 8
RANDOM_STATE = 42

os.makedirs(MODEL_DIR, exist_ok=True)
print("Configuration and imports completed successfully. All warnings suppressed.")

Configuration and imports completed successfully. All warnings suppressed.


In [25]:
def load_data(dataset_path):
    categories = ["covid", "normal"]
    data = []
    labels = []

    for label, category in enumerate(categories):
        path = os.path.join(dataset_path, category)
        if not os.path.exists(path):
            raise FileNotFoundError(f"Dataset folder not found: {path}")

        print(f"Loading: {category}")
        image_count = 0

        for img_name in os.listdir(path):
            img_path = os.path.join(path, img_name)
            if not os.path.isfile(img_path):
                continue

            img = cv2.imread(img_path)
            if img is None:
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = img.astype(np.float32) / 255.0

            data.append(img)
            labels.append(label)
            image_count += 1

        print(f"-> {category}: {image_count} images loaded.")

    if len(data) == 0:
        raise ValueError("No valid images were found in the dataset.")

    return np.array(data, dtype=np.float32), np.array(labels, dtype=np.float32)

print("Loading dataset...")
X, y = load_data(DATASET_PATH)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Training images: {len(X_train)} | Testing images: {len(X_test)}")

Loading dataset...
Loading: covid
-> covid: 69 images loaded.
Loading: normal
-> normal: 25 images loaded.
Training images: 75 | Testing images: 19


In [26]:
def build_resnet():
    base = ResNet50(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    x = GlobalAveragePooling2D()(base.output)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.3)(x)
    output = Dense(1, activation="sigmoid")(x)
    return Model(inputs=base.input, outputs=output)

def build_efficientnet():
    base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    x = GlobalAveragePooling2D()(base.output)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.3)(x)
    output = Dense(1, activation="sigmoid")(x)
    return Model(inputs=base.input, outputs=output)

def build_densenet():
    base = DenseNet121(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    x = GlobalAveragePooling2D()(base.output)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.3)(x)
    output = Dense(1, activation="sigmoid")(x)
    return Model(inputs=base.input, outputs=output)

print("Model builders defined successfully.")

Model builders defined successfully.


In [27]:
model_builders = {
    "resnet": build_resnet,
    "efficientnet": build_efficientnet,
    "densenet": build_densenet
}

results = {}

for name, build_model in model_builders.items():
    print(f"{'='*60}\nTRAINING {name.upper()}\n{'='*60}")
    
    model = build_model()
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    model_path = os.path.join(MODEL_DIR, f"{name}.keras")
    
    checkpoint = ModelCheckpoint(
        model_path, monitor="val_accuracy", mode="max", save_best_only=True, verbose=1
    )
    early_stopping = EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
    )

    model.fit(
        X_train, y_train,
        validation_split=0.20,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[checkpoint, early_stopping],
        verbose=1
    )

    best_model = tf.keras.models.load_model(model_path)
    best_model.compile(optimizer=Adam(learning_rate=1e-4), loss="binary_crossentropy", metrics=["accuracy"])
    
    probabilities = best_model.predict(X_test, verbose=0).ravel()
    predictions = (probabilities >= 0.5).astype(int)

    accuracy = accuracy_score(y_test, predictions)
    results[name] = accuracy

    print(f"\nTest Accuracy ({name.upper()}): {accuracy:.4f}")
    print(classification_report(y_test, predictions, target_names=["COVID", "Normal"], zero_division=0))

TRAINING RESNET
Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 425ms/step - accuracy: 0.7450 - loss: 0.5806
Epoch 1: val_accuracy improved from None to 0.86667, saving model to models\resnet.keras

Epoch 1: finished saving model to models\resnet.keras
8/8 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.6833 - loss: 0.6533 - val_accuracy: 0.8667 - val_loss: 0.4567
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 413ms/step - accuracy: 0.6537 - loss: 0.6654
Epoch 2: val_accuracy did not improve from 0.86667
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 543ms/step - accuracy: 0.6833 - loss: 0.6194 - val_accuracy: 0.8667 - val_loss: 0.4773
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 423ms/step - accuracy: 0.6936 - loss: 0.6271
Epoch 3: val_accuracy did not improve from 0.86667
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 554ms/step - accuracy: 0.6500 - loss: 0.6603 - val_accuracy: 0.8667 - val_loss: 0.4813
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 411ms/step - accuracy: 0.7895 - loss: 0.5418
Epoch 4: val_accuracy did not improve from 0.86667
8/8 ━━━━

In [28]:
best_model_name = max(results, key=results.get)
best_accuracy = results[best_model_name]

source_model_path = os.path.join(MODEL_DIR, f"{best_model_name}.keras")
loaded_best_model = tf.keras.models.load_model(source_model_path)

weights_path = os.path.join(MODEL_DIR, "best_model.weights.h5")
loaded_best_model.save_weights(weights_path)

print(f"\n{'='*60}\nFINAL RESULTS\n{'='*60}")
print(f"BEST MODEL: {best_model_name.upper()} ({best_accuracy * 100:.2f}%)")
print(f"Weights saved successfully to: {weights_path}")


FINAL RESULTS
BEST MODEL: DENSENET (94.74%)
Weights saved successfully to: models\best_model.weights.h5
